# t-Distributed Stochastic Neighbor Embedding (t-SNE)

O PCA tira uma "foto" dos dados a partir de um ângulo bem escolhido. Isso preserva as grandes distâncias, mas é uma projeção **linear**: se os grupos estão separados de um jeito que nenhuma foto plana mostra, eles aparecem sobrepostos. O t-SNE tem outro objetivo. Ele não tenta preservar distâncias, e sim **quem é vizinho de quem**. É como organizar as mesas de uma festa de casamento: amigos próximos precisam sentar na mesma mesa, mas ninguém se importa se a mesa dos primos ficou longe ou perto da mesa dos colegas de trabalho.

Neste notebook vemos um caso em que o PCA não separa os grupos, construímos o t-SNE a partir de três ideias simples, implementamos a classe do zero e assistimos à otimização acontecer. No caminho, aprendemos o que um gráfico de t-SNE mostra e, principalmente, o que ele não mostra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE as SklearnTSNE
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')

## Dataset Sintético

Geramos 400 pontos em **50 dimensões**, divididos em 8 grupos. Os grupos são bem definidos no espaço completo, mas espalhados o bastante para se sobreporem quando olhamos poucas direções de cada vez. As cores mostram os grupos verdadeiros, que o algoritmo não vê.

In [ ]:
X, y = make_blobs(n_samples=400, n_features=50, centers=8, cluster_std=8.0, random_state=42)
X = StandardScaler().fit_transform(X)

X_pca = PCA(n_components=2).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', s=20)
axes[0].set(title='Duas das 50 features originais', xlabel='feature 0', ylabel='feature 1')
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=20)
axes[1].set(title='PCA (2 componentes)', xlabel='PC1', ylabel='PC2')
plt.tight_layout()
plt.show()

Nem as features originais nem a melhor projeção linear separam os 8 grupos. A informação que os distingue está espalhada pelas 50 dimensões, e nenhuma "foto" plana consegue reuni-la.

## Fundamentação Matemática

O t-SNE descreve a vizinhança de cada ponto duas vezes, uma no espaço original e outra no mapa 2D, e move os pontos do mapa até que as duas descrições fiquem parecidas.

### Vizinhança no Espaço Original

Para cada ponto $x_i$, o t-SNE imagina que ele vai "escolher um vizinho" ao acaso, dando mais chance aos pontos mais próximos. A chance de escolher $x_j$ segue uma gaussiana centrada em $x_i$:

$$
p_{j|i} = \frac{\exp\left(-\|x_i - x_j\|^2 / 2\sigma_i^2\right)}{\sum_{k \neq i} \exp\left(-\|x_i - x_k\|^2 / 2\sigma_i^2\right)}
$$

O denominador só faz as chances somarem 1. O detalhe importante é a largura $\sigma_i$, **diferente para cada ponto**, ajustada para que todos tenham aproximadamente o mesmo número efetivo de vizinhos. Esse número é escolhido pelo usuário e se chama **perplexidade**. Um ponto numa região densa ganha um $\sigma_i$ pequeno; um ponto numa região esparsa, um $\sigma_i$ grande.

A função abaixo encontra o $\sigma_i$ por **busca binária**: se o ponto enxerga vizinhos demais, diminui $\sigma$; se enxerga de menos, aumenta. O número efetivo de vizinhos é medido por $2^H$, em que $H$ é a entropia das probabilidades.

In [ ]:
def neighbor_probabilities(sq_distances, perplexity):
    # Probabilidades p_{j|i} de um ponto, dadas as distâncias ao quadrado até os outros
    lo, hi = 1e-3, 1e3
    for _ in range(50):
        sigma = (lo + hi) / 2
        # subtrair o mínimo não muda o resultado normalizado e evita que tudo vire zero
        p = np.exp(-(sq_distances - sq_distances.min()) / (2 * sigma ** 2))
        p = p / p.sum()

        entropy = -np.sum(p * np.log2(p + 1e-12))
        if 2 ** entropy > perplexity:
            hi = sigma      # enxergando vizinhos demais: estreitar a gaussiana
        else:
            lo = sigma      # enxergando vizinhos de menos: alargar a gaussiana
    return p

Para ver o efeito, montamos um exemplo 2D com um grupo **compacto** e um grupo **espalhado** e escolhemos um ponto em cada um, marcado com a estrela. A cor de cada vizinho é a chance de ele ser escolhido. Cada coluna dá um zoom num dos grupos; repare que a escala dos eixos é 5 vezes maior à direita.

In [ ]:
rng = np.random.default_rng(0)
toy = np.vstack([rng.normal(0, 0.5, (100, 2)), rng.normal(0, 2.5, (100, 2)) + [12, 0]])

def plot_neighbors(i, perplexity, size, title):
    # pinta cada ponto com a chance de ser escolhido como vizinho de toy[i]
    sq_dist = np.sum((toy - toy[i]) ** 2, axis=1)
    sq_dist[i] = np.inf
    p = neighbor_probabilities(sq_dist, perplexity)

    plt.scatter(toy[:, 0], toy[:, 1], c=p, cmap='Reds', s=60, edgecolor='gray', lw=0.5)
    plt.scatter(*toy[i], marker='*', s=400, color='black')
    plt.xlim(toy[i, 0] - size, toy[i, 0] + size)
    plt.ylim(toy[i, 1] - size, toy[i, 1] + size)
    plt.gca().set_aspect('equal')
    plt.title(f'{title}, perplexidade = {perplexity}')

plt.figure(figsize=(11, 10))
plt.subplot(2, 2, 1)
plot_neighbors(0, perplexity=5, size=2, title='grupo compacto')
plt.subplot(2, 2, 2)
plot_neighbors(100, perplexity=5, size=10, title='grupo espalhado')
plt.subplot(2, 2, 3)
plot_neighbors(0, perplexity=30, size=2, title='grupo compacto')
plt.subplot(2, 2, 4)
plot_neighbors(100, perplexity=30, size=10, title='grupo espalhado')
plt.tight_layout()
plt.show()

Nos dois grupos, o ponto escolhido distribui sua atenção por mais ou menos o mesmo número de vizinhos, mesmo que no grupo espalhado eles estejam muito mais longe. Com perplexidade maior, cada ponto passa a considerar mais vizinhos.

Por fim, as chances são simetrizadas, para que "$i$ é vizinho de $j$" e "$j$ é vizinho de $i$" tenham o mesmo peso:

$$ p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n} $$

### Vizinhança no Mapa

No mapa 2D, com os pontos em posições $y_i$, a vizinhança é medida da mesma forma, mas com a **distribuição t de Student** no lugar da gaussiana, e é daí que vem o "t" do nome:

$$ q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}} $$

A troca resolve o **problema de aglomeração** (*crowding*). Em 50 dimensões há espaço de sobra para um ponto ter dezenas de vizinhos a uma distância média; em 2D não cabe tanta gente a essa mesma distância, e os pontos acabariam esmagados no centro do mapa. A t de Student tem **cauda pesada**, decaindo bem mais devagar que a gaussiana, e por isso permite que pontos moderadamente distantes fiquem muito mais longe no mapa, o que abre espaço entre os grupos.

In [ ]:
d = np.linspace(0, 5, 200)

plt.figure(figsize=(8, 4))
plt.plot(d, np.exp(-d ** 2), label='gaussiana (espaço original)', lw=2)
plt.plot(d, 1 / (1 + d ** 2), label='t de Student (mapa 2D)', lw=2)
plt.xlabel('distância')
plt.ylabel('similaridade')
plt.title('A t de Student tem cauda pesada')
plt.legend()
plt.show()

### Otimização

A diferença entre as vizinhanças originais $P$ e as do mapa $Q$ é medida pela **divergência de Kullback-Leibler**,

$$ C = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}} $$

e os pontos do mapa são movidos por **descida de gradiente** para diminuí-la. O gradiente tem uma leitura física simples:

$$ \frac{\partial C}{\partial y_i} = 4 \sum_{j} (p_{ij} - q_{ij})\,(1 + \|y_i - y_j\|^2)^{-1}\,(y_i - y_j) $$

Cada par de pontos funciona como uma **mola**. Quando $p_{ij} > q_{ij}$, os pontos eram vizinhos mas estão longe no mapa, e a mola os **atrai**; quando $p_{ij} < q_{ij}$, não eram vizinhos mas estão perto, e a mola os **repele**. A KL pune muito mais separar vizinhos do que aproximar estranhos, porque o termo é multiplicado por $p_{ij}$, e é por isso que o t-SNE preserva bem a estrutura **local** e não se compromete com a global.

Um último truque é o **exagero inicial**: nas primeiras 250 iterações, $P$ é multiplicado por 4. A atração fica mais forte no começo e os vizinhos se juntam antes que a repulsão passe a organizar o mapa.

## Implementação

A classe junta as três partes: calcula $P$ uma única vez e, a cada iteração, calcula $Q$, o gradiente e dá um passo de descida. A cada 10 iterações guardamos uma cópia do mapa em `history`, para depois assistir à otimização.

In [ ]:
class TSNE:
    def __init__(self, n_components=2, perplexity=30, learning_rate=500, n_iter=1000, random_state=None):
        self.n_components = n_components
        self.perplexity = perplexity
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.random_state = random_state

    def fit_transform(self, X):
        n = len(X)

        # vizinhanças no espaço original
        sq_distances = np.sum((X[:, None, :] - X[None, :, :]) ** 2, axis=-1)
        np.fill_diagonal(sq_distances, np.inf)                # um ponto não é vizinho de si mesmo
        P = np.array([neighbor_probabilities(d, self.perplexity) for d in sq_distances])
        P = (P + P.T) / (2 * n)

        rng = np.random.default_rng(self.random_state)
        Y = rng.normal(0, 1e-4, (n, self.n_components))   # mapa inicial aleatório
        self.history = []

        for it in range(self.n_iter):
            exaggeration = 4 if it < 250 else 1

            # vizinhanças no mapa (t de Student)
            diff = Y[:, None, :] - Y[None, :, :]               # y_i - y_j para todos os pares
            student = 1 / (1 + np.sum(diff ** 2, axis=-1))
            np.fill_diagonal(student, 0)
            Q = student / student.sum()

            # gradiente: soma das forças de mola sobre cada ponto
            forces = (exaggeration * P - Q) * student
            grad = 4 * np.sum(forces[:, :, None] * diff, axis=1)
            Y = Y - self.learning_rate * grad

            if it % 10 == 0:
                self.history.append(Y.copy())

        return Y

In [ ]:
tsne = TSNE(perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=20)
axes[0].set_title('PCA')
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', s=20)
axes[1].set_title('t-SNE')
plt.tight_layout()
plt.show()

Os 8 grupos aparecem completamente separados, e o t-SNE não usou as cores para isso, apenas as distâncias entre os pontos. Como guardamos o mapa a cada 10 iterações, podemos ver como ele evolui.

In [ ]:
iterations = [0, 20, 50, 240, 400, 990]

fig, axes = plt.subplots(1, len(iterations), figsize=(20, 3.5))
for ax, it in zip(axes, iterations):
    Y_it = tsne.history[it // 10]
    ax.scatter(Y_it[:, 0], Y_it[:, 1], c=y, cmap='tab10', s=6)
    ax.set(title=f'iteração {it}', xticks=[], yticks=[])
plt.tight_layout()
plt.show()

Na iteração 0 os pontos estão em posições aleatórias, todos misturados. Durante o exagero inicial a atração domina e os vizinhos vão se juntando: na iteração 240 os grupos já são visíveis, embora ainda próximos uns dos outros e com pontos soltos entre eles. Quando o exagero acaba, na iteração 250, a repulsão ganha espaço, o mapa se expande e os grupos se separam por completo.

### Comparação com o Scikit-Learn

```python
from sklearn.manifold import TSNE

TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)
```

A versão da biblioteca usa a aproximação de Barnes-Hut, que a torna muito mais rápida, e inicializa o mapa com PCA em vez de posições aleatórias. Os parâmetros principais são `perplexity` (padrão 30), `n_components` e `random_state`, e o atributo `kl_divergence_` guarda o custo final da otimização.

In [ ]:
X_sk = SklearnTSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, Z, title in [(axes[0], X_tsne, 'Nossa implementação'), (axes[1], X_sk, 'Scikit-Learn')]:
    ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap='tab10', s=20)
    ax.set_title(title)
plt.tight_layout()
plt.show()

Os mapas diferem na posição e na orientação dos grupos, e isso é esperado: o custo depende apenas de quem está perto de quem, então girar, espelhar ou mover o mapa inteiro não muda nada. O que precisa coincidir é quais pontos ficaram juntos, e nos dois casos são os mesmos 8 grupos.

## Perplexidade

A perplexidade é o principal hiperparâmetro do método, o número aproximado de vizinhos que cada ponto considera.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, perplexity in zip(axes, [2, 30, 100]):
    Z = TSNE(perplexity=perplexity, random_state=42).fit_transform(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap='tab10', s=15)
    ax.set_title(f'perplexidade = {perplexity}')
plt.tight_layout()
plt.show()

Com perplexidade 2, cada ponto olha para apenas um ou dois vizinhos, e o mapa vira um conjunto de pequenos fragmentos que quebram os grupos verdadeiros. Com 30, dentro da faixa usual de 5 a 50, os grupos aparecem bem. Com 100, cada ponto considera muitos vizinhos e o resultado fica mais "global", com os grupos mais próximos entre si. Não existe valor certo: na prática, vale testar alguns e só confiar nas estruturas que aparecem em todos.

## Como (Não) Ler um Gráfico t-SNE

Num gráfico de dispersão comum, tamanho e distância têm significado; no t-SNE, não necessariamente. Para ver isso, montamos três grupos em 2D com propriedades conhecidas: A e C são compactos e B é **10 vezes mais espalhado**, enquanto C está **4 vezes mais longe** de B do que A está.

In [ ]:
rng = np.random.default_rng(0)
A = rng.normal(0, 0.5, (100, 2))
B = rng.normal(0, 5.0, (100, 2)) + [40, 0]
C = rng.normal(0, 0.5, (100, 2)) + [200, 0]
X_abc = np.vstack([A, B, C])
labels = np.repeat([0, 1, 2], 100)

Z_abc = SklearnTSNE(perplexity=30, random_state=42).fit_transform(X_abc)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, Z, title in [(axes[0], X_abc, 'Dados originais'), (axes[1], Z_abc, 't-SNE')]:
    for k, name in enumerate('ABC'):
        group = Z[labels == k]
        ax.scatter(group[:, 0], group[:, 1], s=12)
        ax.text(group[:, 0].mean(), group[:, 1].max(), name, fontsize=18, weight='bold',
                ha='center', va='bottom')
    ax.set_title(title)
plt.tight_layout()
plt.show()

No mapa, os três grupos têm **o mesmo tamanho** e estão a **distâncias parecidas**. O tamanho se perde porque cada ponto ajusta seu $\sigma_i$ para enxergar o mesmo número de vizinhos, então um grupo espalhado e um compacto produzem vizinhanças idênticas. A distância entre grupos se perde porque a KL quase não penaliza aproximar pontos distantes, e nada obriga o mapa a manter C mais longe.

Num gráfico de t-SNE, a informação confiável é **quais pontos estão juntos**. O tamanho dos grupos, a distância entre eles e os eixos não têm significado.

## Aplicação: Dígitos Manuscritos

O dataset `digits` tem 1797 imagens de 8×8 pixels, ou seja, cada imagem é um ponto em 64 dimensões.

In [ ]:
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)
y_digits = digits.target

fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for ax, img, label in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(label)
    ax.axis('off')
plt.show()

In [ ]:
Z_pca = PCA(n_components=2).fit_transform(X_digits)
Z_tsne = SklearnTSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_digits)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, Z, title in [(axes[0], Z_pca, 'PCA'), (axes[1], Z_tsne, 't-SNE')]:
    ax.scatter(Z[:, 0], Z[:, 1], c=y_digits, cmap='tab10', s=8, alpha=0.7)
    for digit in range(10):
        # escreve o dígito no centro do seu grupo
        center = np.median(Z[y_digits == digit], axis=0)
        ax.text(*center, str(digit), fontsize=20, weight='bold', ha='center', va='center',
                bbox=dict(boxstyle='circle', facecolor='white', alpha=0.8))
    ax.set(title=title, xticks=[], yticks=[])
plt.tight_layout()
plt.show()

No PCA, vários dígitos se sobrepõem no centro do gráfico. No t-SNE, cada dígito forma seu próprio grupo, e os poucos pontos fora do lugar costumam ser dígitos escritos de forma ambígua, como um 1 que parece 7.

## Custo e Limitações

A versão exata compara todos os pares de pontos e custa $O(n^2)$ por iteração, em tempo e em memória. A aproximação de Barnes-Hut do Scikit-Learn reduz bastante esse custo, mas o método ainda fica pesado acima de algumas dezenas de milhares de pontos. Uma prática comum é reduzir os dados para cerca de 50 dimensões com PCA antes, o que acelera o cálculo e remove ruído.

A limitação mais restritiva é que **não existe `transform`**. O t-SNE otimiza as posições dos pontos que recebeu, e incluir um ponto novo exige rodar tudo de novo. Por isso ele é uma ferramenta de visualização, e não de extração de features: os eixos não significam nada e não há equivalente à variância explicada do PCA. O resultado também depende da semente e da perplexidade, então convém fixar `random_state` e comparar mais de um valor de perplexidade antes de concluir qualquer coisa.

## Exercícios

### Exercício 1: Sensibilidade à Semente

Rode a nossa implementação sobre o dataset sintético com quatro valores diferentes de `random_state` e plote os mapas lado a lado. Os mapas são iguais? E os grupos formados? Explique o que isso diz sobre interpretar a posição de um grupo no gráfico.

In [ ]:
# Seu código aqui

### Exercício 2: Sem Exagero Inicial

Crie uma versão da classe `TSNE` em que o fator de exagero seja sempre 1 e compare com a original, plotando os mapas finais e algumas etapas de `history`. Em que iteração os grupos começam a aparecer em cada caso? A separação final mudou?

In [ ]:
# Seu código aqui

### Exercício 3: Vinhos

Carregue o dataset `load_wine`, padronize as features e projete em 2D com PCA e com t-SNE, testando perplexidades 5, 30 e 50. Compare a separação das 3 classes. Com apenas 178 amostras e 13 features, o t-SNE traz alguma vantagem sobre o PCA?

In [ ]:
# Seu código aqui